In [32]:
import pandas as pd

ARTICLE_ID_COL = "#"

# Read from your sources (local or Google Sheets) into `articles` and `editors` first
articles = pd.read_excel("https://docs.google.com/spreadsheets/d/1oBK2KbCVNEbxrZoz8HvkQ74Yo8B_RI3zz1vLMLUawY0/export?format=xlsx", engine="openpyxl")
editors = pd.read_excel("https://docs.google.com/spreadsheets/d/1xjsUsxE_x7uEMYhv_BEXzsDS_Rk_bRk-lrRf-Fiit98/export?format=xlsx", engine="openpyxl")

# Normalize
articles["Category"] = articles["Category"].str.strip().str.lower()
articles["Region"]   = articles["Region"].str.strip().str.lower()
articles["Reviewer 1"] = articles["Reviewer 1"].astype("string").str.strip()

for col in ["Topics", "Regions"]:
    editors[col] = editors[col].fillna("").apply(
        lambda s: [x.strip().lower() for x in s.split(",") if x.strip()]
    )

# Count existing Reviewer 1 load
current_load = {e: 0 for e in editors["Your Name"]}
existing_counts = (
    articles["Reviewer 1"].dropna().astype(str).value_counts().to_dict()
)
for ed, c in existing_counts.items():
    if ed in current_load:
        current_load[ed] += c

# Decide max_load (roughly equal)
num_articles = len(articles)
num_editors  = len(editors)

if "max_load" in editors.columns:
    max_load = editors.set_index("Your Name")["max_load"].to_dict()
else:
    base = num_articles // num_editors
    extra = num_articles % num_editors
    names = editors["Your Name"].tolist()
    max_load = {
        name: base + (1 if i < extra else 0)
        for i, name in enumerate(names)
    }

# Ensure no one is over their max due to historical load
for ed in current_load:
    max_load[ed] = max(max_load.get(ed, 0), current_load[ed])

# Cross join Articles × Editors
pairs = pd.merge(
    articles[[ARTICLE_ID_COL, "Category", "Region"]].assign(key=1),
    editors.assign(key=1),
    on="key"
).drop(columns=["key"])

def score_row(row):
    theme_match  = 1 if row["Category"] in row["Topics"] else 0
    region_match = 1 if row["Region"] in row["Regions"] else 0
    return 2 * theme_match + 1 * region_match

pairs["score"] = pairs.apply(score_row, axis=1)


In [33]:
for idx, row in articles.iterrows():
    # Skip rows that are clearly empty / not real articles
    title = str(row["Title"]).strip() if pd.notna(row["Title"]) else ""
    if title == "":
        continue

    article_id = row[ARTICLE_ID_COL]
    r1 = str(row["Reviewer 1"]).strip() if pd.notna(row["Reviewer 1"]) else ""

    # Skip if Reviewer 1 is already assigned
    if r1 != "" and r1.lower() != "<na>":
        continue

    g = pairs[pairs[ARTICLE_ID_COL] == article_id].copy()
    g["curr_load"] = g["Your Name"].map(current_load)

    # Sort: best score first, then least loaded
    g = g.sort_values(["score", "curr_load"], ascending=[False, True])

    chosen = None

    # Prefer positive score and within capacity
    for _, r in g.iterrows():
        ed = r["Your Name"]
        if r["score"] <= 0:
            break
        if current_load[ed] < max_load[ed]:
            chosen = ed
            break

    # Fallback: least loaded editor with capacity
    if chosen is None:
        cap = g[g["Your Name"].map(lambda e: current_load[e] < max_load[e])]
        if not cap.empty:
            chosen = cap.sort_values("curr_load").iloc[0]["Your Name"]
        else:
            # Last resort: globally least loaded
            chosen = min(current_load, key=current_load.get)

    articles.at[idx, "Reviewer 1"] = chosen
    current_load[chosen] += 1

articles = articles[~pd.isna(articles['Title'])]

In [34]:
articles.to_excel('output/assigned_editors_one.xlsx', index=False)

In [ ]:
#######################

In [14]:
empty_title_with_reviewer = articles[
    (articles["Title"].isna() | (articles["Title"].astype(str).str.strip() == "")) &
    (articles["Reviewer 1"].notna()) &
    (articles["Reviewer 1"].astype(str).str.strip() != "")
]

print("Rows with empty Title but a reviewer assigned:", len(empty_title_with_reviewer))

Rows with empty Title but a reviewer assigned: 0


In [17]:
real = articles[~pd.isna(articles["Title"].astype(str).str.strip())]

no_reviewer = real[
    real["Reviewer 1"].isna() |
    (real["Reviewer 1"].astype(str).str.strip() == "") |
    (real["Reviewer 1"].astype(str).str.lower() == "<na>")
]

print("Real articles without a reviewer:", len(no_reviewer))


Real articles without a reviewer: 0


In [ ]:
load = real["Reviewer 1"].astype(str).str.strip().value_counts()
print("\nReviewer 1 load per editor:")
print(load)

print("\nLoad stats:")
print(load.describe())


Reviewer 1 load per editor:
Reviewer 1
Alexandra Helfgott    11
Hekmat Aboukhater     10
Vincient Arnold        9
Catherine Dai          9
Mike Henry             9
Rachel Lee             8
Emily D’Arco           8
Sarah Cheung           8
Jacob Coughlin         8
Jonas Nepozitek        8
Kevin Chen             8
Richard Ulbricht       8
Michael Iwuoha         8
Lisa de Rafols         8
Gabriel Falk           8
Rafael Gonzalez        8
Name: count, dtype: int64

Load stats:
count    16.000000
mean      8.500000
std       0.894427
min       8.000000
25%       8.000000
50%       8.000000
75%       9.000000
max      11.000000
Name: count, dtype: float64
